In [1]:
!pip install requests beautifulsoup4 pandas lxml

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import requests

url = "https://books.toscrape.com/"
response = requests.get(url)

print(response.status_code)
print(len(response.text))

200
51294


In [3]:
from bs4 import BeautifulSoup

response.encoding = "utf-8"
soup = BeautifulSoup(response.text, "lxml")

books = soup.find_all("article", class_="product_pod")
print(len(books))

20


In [4]:
print(books[0].prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [5]:
book = books[0]

title = book.h3.a["title"]
price = book.find("p", class_="price_color").text
availability = book.find("p", class_="instock availability").text.strip()
rating = book.p["class"][1]

print(title)
print(price)
print(availability)
print(rating)

A Light in the Attic
£51.77
In stock
Three


In [6]:
import pandas as pd

rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
data = []

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text.strip()
    availability = book.find("p", class_="instock availability").text.strip()
    rating_word = book.p["class"][1]
    rating = rating_map.get(rating_word, None)
    
    data.append({
        "title": title,
        "price": price,
        "availability": availability,
        "rating": rating
    })

df = pd.DataFrame(data)
df.head(20)

,title,price,availability,rating
0,A Light in the Attic,£51.77,In stock,3
1,Tipping the Velvet,£53.74,In stock,1
2,Soumission,£50.10,In stock,1
3,Sharp Objects,£47.82,In stock,4
4,Sapiens: A Brief History of Humankind,£54.23,In stock,5
5,The Requiem Red,£22.65,In stock,1
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,In stock,4
7,The Coming Woman: A Novel Based on the Life of...,£17.93,In stock,3
8,The Boys in the Boat: Nine Americans and Their...,£22.60,In stock,4
9,The Black Maria,£52.15,In stock,1


In [9]:
import time

all_data = []
base_url = "https://books.toscrape.com/catalogue/page-{}.html"

for page_num in range(1, 51):
    url = base_url.format(page_num)
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Stopped at page {page_num}, status {response.status_code}")
        break
    
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "lxml")
    books = soup.find_all("article", class_="product_pod")
    
    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text.strip()
        availability = book.find("p", class_="instock availability").text.strip()
        rating_word = book.p["class"][1]
        rating = rating_map.get(rating_word, None)
        
        all_data.append({
            "title": title,
            "price": price,
            "availability": availability,
            "rating": rating
        })
    
    time.sleep(0.5)  # be polite to the server

df_all = pd.DataFrame(all_data)
print(len(df_all))
df_all.head()

1000


,title,price,availability,rating
0,A Light in the Attic,£51.77,In stock,3
1,Tipping the Velvet,£53.74,In stock,1
2,Soumission,£50.10,In stock,1
3,Sharp Objects,£47.82,In stock,4
4,Sapiens: A Brief History of Humankind,£54.23,In stock,5


In [10]:
df_all.to_csv("books_data.csv", index=False)
print("Saved:", len(df_all), "rows")

Saved: 1000 rows


In [11]:
df_all.head()

,title,price,availability,rating
0,A Light in the Attic,£51.77,In stock,3
1,Tipping the Velvet,£53.74,In stock,1
2,Soumission,£50.10,In stock,1
3,Sharp Objects,£47.82,In stock,4
4,Sapiens: A Brief History of Humankind,£54.23,In stock,5
